# Evaluate splits

Observed that there are many diseases which are not present in the test set - due to the split function - something weird is going on. Investigate why.

In [123]:
import sys
sys.path.append("../..")
from src.training import helpers as tr_h
import scanpy as sc
import numpy as np

data_paths = ["/aloy/home/ddalton/projects/scGPT_playground/data/pp_data-25-08-14-01/data.h5ad"]

def filter_adata_by_genes(adata, mask_genes):

    # mask the genes
    adata = adata[:, mask_genes]
    print(f"adata shape after gene filtering: {adata.shape}")
    # mask samples
    # non_nan_percentage = np.sum(~np.isnan(adata.X), axis=1) / adata.X.shape[1]
    non_zero_non_nan_mask = ~np.isnan(adata.X) & ~(adata.X == 0)

    non_zero_non_nan_mask_pct = np.sum(non_zero_non_nan_mask, axis=1) / adata.X.shape[1]

    # mask samples that have less than 30% non-NaN values
    mask_samples = non_zero_non_nan_mask_pct >= 0.3
    print(
        f"Filtering out {np.sum(~mask_samples)} / {len(mask_samples)} samples with less than 30% non-NaN values"
    )

    # apply the mask to the AnnData object
    adata = adata[mask_samples, :]
    print(f"adata shape after sample filtering: {adata.shape}")

    return adata    

def clean_adata_qc(adata:sc.AnnData, disease_label:str="celltype", n_samples:int=2, n_dt:int=2)->sc.AnnData:
    """Same criteria as in PP scritps
    + n_samples per dataset
    + n_dt per disease
    """
    # filter by sufficient samples
    if "Control" in adata.obs[disease_label].unique():
        # split adata into control and disease
        adata_control = adata.obs[adata.obs[disease_label]=="Control"]
        adata_dis = adata.obs[adata.obs[disease_label]!="Control"]

        # check which datasets have enough samples
        _datasets_control_passed = [k for k, v in dict(adata_control.groupby("dataset", observed=True)['celltype'].count()).items() if v>=n_samples ]
        print(f"Nº of datasets with +{n_samples} control samples: {len(_datasets_control_passed)}")

        _datasets_dis_passed = [k for k, v in dict(adata_dis.groupby("dataset", observed=True)['celltype'].count()).items() if v>=n_samples ]
        print(f"Nº of datasets with +{n_samples} disease samples: {len(_datasets_dis_passed)}")

        _datasets_passed = set(_datasets_control_passed).intersection(set(_datasets_dis_passed))
        print(f"Nº of datasets with +{n_samples} samples (control and disease): {len(_datasets_passed)}")

        # filter adata
        adata = adata[adata.obs["dataset"].isin(_datasets_passed)]
        print(f"adata shape after filtering datasets with +{n_samples} samples: {adata.shape}")
    
    else:
        _datasets_passed = [k for k, v in dict(adata.groupby("dataset", observed=True)['celltype'].count()).items() if v>=n_samples ]
        print(f"Nº of datasets with +{n_samples} samples (disease): {len(_datasets_passed)}")

        # filter adata
        adata = adata[adata.obs["dataset"].isin(_datasets_passed)]
        print(f"adata shape after filtering datasets with +{n_samples} samples: {adata.shape}")


    # filter by sufficient datasets
    dis = adata.obs["do_id"].unique()
    dis = [d for d in dis if d != "Control"]  # remove controls
    _passed_diseases = list()
    for d in dis:
        _df_counts = adata.obs[adata.obs["do_id"] == d].groupby("dataset", observed=True).size()
        if len(_df_counts) >= 2:
            _passed_diseases.append(d)
    print(f"Nº of passed diseases {len(_passed_diseases)}/ {len(dis)}")
    adata = adata[adata.obs["do_id"].isin(_passed_diseases)]
    return adata


adata = sc.read(data_paths[0])
print("adata loaded")
print(adata.shape)

print(adata.obs.columns)
# adata.obs["celltype"] = adata.obs["do_term"].astype("category")
adata.obs["celltype"] = adata.obs["do_id"].astype("category")

# generate celltype label
celltype_id_labels = adata.obs["celltype"].astype("category").cat.codes.values
celltypes = adata.obs["celltype"].unique()
num_types = len(np.unique(celltype_id_labels))
id2type = dict(enumerate(adata.obs["celltype"].astype("category").cat.categories))
adata.obs["celltype_id"] = celltype_id_labels

# config parameters
data_is_raw = True
filter_gene_by_counts = False

mask_genes = tr_h.get_top_k_most_present_genes(
    adata, k=3501
)
print(f"Combined mask {np.sum(mask_genes)} genes left")

# apply gene masking
adata = filter_adata_by_genes(adata, mask_genes)
print(adata.shape)
# clean (enough samples and datasets)
adata = clean_adata_qc(adata)
print(adata.shape)

adata loaded
(35626, 20608)
Index(['ids', 'dataset', 'dataset_id', 'batch', 'batch_id', 'dsaid', 'tissue',
       'n_genes', 'disease', 'celltype', 'disease_study', 'library',
       'doid_study', 'doid_id', 'do_id', 'doid_disease'],
      dtype='object')


/home/ddalton/Data/old_home/miniconda3/envs/scgpt_2/lib/python3.12/site-packages/numpy/lib/nanfunctions.py:1879: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


Combined mask 3501 genes left
adata shape after gene filtering: (35626, 3501)
Filtering out 581 / 35626 samples with less than 30% non-NaN values
adata shape after sample filtering: (35045, 3501)
(35045, 3501)
Nº of datasets with +2 control samples: 650
Nº of datasets with +2 disease samples: 649
Nº of datasets with +2 samples (control and disease): 648
adata shape after filtering datasets with +2 samples: (35030, 3501)
Nº of passed diseases 130/ 131
(20934, 3501)


In [ ]:

def apply_combat_adata(adata:sc.AnnData)->sc.AnnData:
    # copy data
    adata_tmp = adata.copy()
    X = adata_tmp.X.copy()

    # mask nans
    mask_nans = np.isnan(X)

    # compute medians, ignoring NaNs
    col_medians = np.nanmedian(X, axis=0)

    # broadcast to fill NaNs with the corresponding gene's median
    X = np.where(np.isnan(X), col_medians, X)
    adata_tmp.X = X

    # apply batch correction
    X = sc.pp.combat(adata_tmp, key="dataset", inplace=False)

    # restore nans
    X[mask_nans] = np.nan

    adata_tmp.X = X
    return adata_tmp


adata_2 = apply_combat_adata(adata)

/home/ddalton/Data/old_home/miniconda3/envs/scgpt_2/lib/python3.12/site-packages/scanpy/preprocessing/_combat.py:351: RuntimeWarning: divide by zero encountered in divide
  (abs(g_new - g_old) / g_old).max(), (abs(d_new - d_old) / d_old).max()


In [132]:
adata_2.X

array([[4.73889167, 5.98489608, 0.81345383, ..., 4.49040328, 5.70208277,
        5.039639  ],
       [4.1581636 , 4.81012234, 2.60815235, ..., 5.20813649, 5.11907497,
        2.75186012],
       [4.84135536, 4.27941261, 2.13671516, ..., 4.0735401 , 5.35986583,
        2.58292091],
       ...,
       [5.15701621, 5.55571926, 2.65672726, ..., 5.47021996, 6.5699149 ,
        2.64844302],
       [4.43432219, 4.13440668, 1.91554755, ..., 4.99913006, 5.73720134,
        2.95446122],
       [4.29697955, 4.83603037, 1.42250369, ..., 4.87578922, 5.93639824,
        2.81203075]])